# Fine-tuning whisper-small Bahasa Jawa (GPU Colab / Kaggle)

Menjalankan baseline `small-pretrained`, pencarian learning rate (fold 1), dan fine-tuning 5 fold
`small-ft` dari `configs/experiments.yaml`.

Sebelum mulai:
1. Kode eksperimen harus sudah **di-push** ke GitHub, di branch `feat/experiments` atau `main`
   (notebook ini meng-clone repo; sesuaikan `BRANCH` di sel clone).
2. Colab: *Runtime → Change runtime type → T4 GPU*. Kaggle: *Settings → Accelerator → GPU*, internet aktif.
3. Dengan `USE_DRIVE = True` siapkan ±9 GB ruang kosong di Google Drive (8 model small).

Perkiraan waktu pada T4: 2–4 jam (belum diukur). Jika sesi terputus, jalankan ulang semua sel:
fold yang sudah selesai dilewati.

In [ ]:
!nvidia-smi

In [ ]:
import os
WORKDIR = "/kaggle/working" if os.path.exists("/kaggle/working") else "/content"
BRANCH = "feat/experiments"  # ganti ke "main" setelah branch ini di-merge
%cd {WORKDIR}
![ -d korpus-asr-jawa-whisper ] || git clone -b {BRANCH} https://github.com/jonathankenan/korpus-asr-jawa-whisper.git
%cd korpus-asr-jawa-whisper
assert os.path.exists("src/run_experiments.py"), "Kode eksperimen belum di-push ke GitHub."
!git log -1 --oneline

In [ ]:
!pip install -q -r requirements.txt

## Simpan hasil ke Google Drive (khusus Colab)

Tanpa ini, model dan prediksi hilang saat sesi berakhir. Di Kaggle biarkan `False`
(keluaran di `/kaggle/working` tersimpan saat *Save Version*).

In [ ]:
USE_DRIVE = os.path.exists("/content") and not os.path.exists("/kaggle/working")

if USE_DRIVE:
    import shutil
    from pathlib import Path
    from google.colab import drive
    drive.mount("/content/drive")
    base = Path("/content/drive/MyDrive/whisper-jv")
    for sub in ["models", "results/predictions", "results/training"]:
        target, local = base / sub, Path(sub)
        target.mkdir(parents=True, exist_ok=True)
        if local.is_symlink():
            continue
        if local.exists():  # salin file dari repo yang belum ada di Drive
            for f in local.rglob("*"):
                dest = target / f.relative_to(local)
                if f.is_file() and not dest.exists():
                    dest.parent.mkdir(parents=True, exist_ok=True)
                    shutil.copy2(f, dest)
            shutil.rmtree(local)
        local.parent.mkdir(parents=True, exist_ok=True)
        local.symlink_to(target, target_is_directory=True)
        print(local, "->", target)

## Data dan model

In [ ]:
!bash scripts/fetch_data.sh
!python scripts/download_models.py openai/whisper-small

## Uji cepat (1–2 menit): pastikan pelatihan jalan di GPU

In [ ]:
!python src/train.py --exp zz-uji --fold 1 --max-train 32 --max-val 8 --epochs 1 --warmup-steps 2
!rm -rf models/zz-uji results/training/zz-uji

## Baseline pretrained small

In [ ]:
!python src/run_experiments.py small-pretrained

## Fine-tuning

Tiga learning rate pada fold 1, lalu 5 fold dengan learning rate ber-WER **validation** terkecil.

In [ ]:
!python src/run_experiments.py small-ft-lr5e-6 small-ft-lr1e-5 small-ft-lr3e-5 small-ft

## Hasil

In [ ]:
!python src/evaluate.py
from IPython.display import Markdown
Markdown(open("results/summary.md", encoding="utf-8").read())

In [ ]:
!zip -r -q {WORKDIR}/hasil_gpu.zip results/predictions/small-* results/training/small-* results/eval/small-* results/analysis/small-*
if USE_DRIVE:
    from google.colab import files
    files.download(f"{WORKDIR}/hasil_gpu.zip")

## Setelah selesai

1. Salin `hasil_gpu.zip` ke akar repo di laptop, lalu:
   ```
   Expand-Archive hasil_gpu.zip -DestinationPath . -Force
   python src/evaluate.py
   python src/analyze_errors.py --exp small-ft --baseline small-pretrained
   ```
2. Model ada di `Drive/whisper-jv/models/small-ft/fold1..5` (fold yang memakai ulang run pencarian
   learning rate ada di folder run tersebut). Bagikan tautannya dan catat di `docs/MODEL.md`.